# The annotation assignment vs. the songs that were removed from the corpus

**Goal.** `asignacion_versos_por_grupo.xlsx` is the working copy handed to the four annotation
groups: one sheet per group, one row per verse. The files in `data/old_data/` record the songs
that earlier pipeline stages *discarded* — empty lyrics, duplicate scrapes, wrong language. A
discarded song has no business appearing in the assignment, so every one of those files gives us
a list of `song_id`s that should be **absent** from the workbook.

`04_spanish_predicted_songs.csv` is the exception: it is the *kept* side of the language filter,
so its songs are the ones that should be **present**.

The notebook does two things:

1. flattens the four sheets into a single dataframe (the group survives as a column, not as a sheet);
2. runs one section per CSV in `data/old_data/`, checking the direction of membership that file implies.

In [21]:
import re
from pathlib import Path

import pandas as pd

DATA = Path("../data") if Path("../data").exists() else Path("data")
WORKBOOK = DATA / "asignacion_versos_por_grupo.xlsx"
OLD = DATA / "old_data"

pd.set_option("display.max_colwidth", 60)

## 1. The four sheets as one dataframe

`pd.read_excel(..., sheet_name=None)` returns a dict of sheet name → frame. The sheet name *is*
the group label, so it moves into a `group` column and the dict collapses into one frame. Nothing
downstream has to know the workbook had tabs.

In [22]:
sheets = pd.read_excel(WORKBOOK, sheet_name=None)
print("sheets in the workbook:", list(sheets))

assignment = pd.concat(
    [frame.assign(group=name) for name, frame in sheets.items()],
    ignore_index=True,
)
assignment["song_id"] = assignment["song_id"].astype("int64")

# Every sheet must carry the same columns, or the concat silently padded rows with NaN.
columns = {name: tuple(frame.columns) for name, frame in sheets.items()}
assert len(set(columns.values())) == 1, f"sheets disagree on their columns: {columns}"
assert not assignment.isna().any(axis=None), "NaN in the flattened frame"

print(f"\n{len(assignment):,} verse rows, {assignment['song_id'].nunique():,} distinct songs")
assignment.head()

sheets in the workbook: ['Grupo 1', 'Grupo 2', 'Grupo 3', 'Grupo 4']

24,042 verse rows, 3,129 distinct songs


,song_id,artist,title,verse_id,verse_number,total_verses,verse_text,group
0,4175,Daddy Yankee,Rompe,4175__1,1,8,"[Letra de ""Rompe""]\n\n[Intro]\nYou know\nLos capos están...",Grupo 1
1,4175,Daddy Yankee,Rompe,4175__2,2,8,"[Coro]\nRompe, rompe, rompe, bien guilla'o\nRompe, rompe...",Grupo 1
2,4175,Daddy Yankee,Rompe,4175__3,3,8,"[Verso 1]\nMy boo, no se limita a la hora de romper su p...",Grupo 1
3,4175,Daddy Yankee,Rompe,4175__4,4,8,"[Coro]\nRompe, rompe, rompe, bien guilla'o\nRompe, rompe...",Grupo 1
4,4175,Daddy Yankee,Rompe,4175__5,5,8,"[Puente]\nVoy chillin', tranquilo, that's right\nBuscand...",Grupo 1


In [23]:
assignment.groupby("group").agg(
    verses=("verse_id", "size"),
    songs=("song_id", "nunique"),
    artists=("artist", "nunique"),
)

,verses,songs,artists
group,,,
Grupo 1,6093,783,469
Grupo 2,6042,782,467
Grupo 3,6034,782,448
Grupo 4,5873,782,464


The four groups are balanced — the assignment was built to give each group roughly the same load.

One structural check before the membership tests, since the rest of the notebook keys on
`song_id`: is a song ever split across two groups? If it were, "is this song in the workbook"
would stop being a clean yes/no per group.

In [24]:
per_song_groups = assignment.groupby("song_id")["group"].nunique()
split_songs = per_song_groups[per_song_groups > 1]
print(f"songs appearing in more than one group: {len(split_songs)}")

duplicated_verses = assignment[assignment["verse_id"].duplicated(keep=False)]
print(f"repeated verse_id rows: {len(duplicated_verses)} "
      f"({duplicated_verses['verse_id'].nunique()} ids, "
      f"{duplicated_verses['song_id'].nunique()} songs)")

songs appearing in more than one group: 0
repeated verse_id rows: 302 (151 ids, 27 songs)


No song crosses a group boundary, so `song_id` is a safe key.

The repeated `verse_id`s are a separate matter and worth flagging: the same id appears twice
*inside a single group*, and the two rows carry **different** text. That is an id collision in the
workbook, not a song-selection problem, so it is out of scope here — but it is real, and the
cell below shows the shape of it.

In [25]:
if len(duplicated_verses):
    disagreeing = (
        duplicated_verses.groupby("verse_id")["verse_text"].nunique().gt(1).sum()
    )
    print(f"{disagreeing} of the {duplicated_verses['verse_id'].nunique()} repeated ids "
          f"carry two different verse texts")
    display(duplicated_verses.sort_values("verse_id")
            .head(4)[["group", "song_id", "artist", "title", "verse_id", "verse_text"]])

151 of the 151 repeated ids carry two different verse texts


,group,song_id,artist,title,verse_id,verse_text
6812,Grupo 2,1018992,Hombres G,Marta Tiene Un Marcapasos,1018992__1,[Instrumental]
6813,Grupo 2,1018992,Hombres G,Marta Tiene Un Marcapasos,1018992__1,Marta tiene un marcapasos que le anima el corazón\nNo ti...
6814,Grupo 2,1018992,Hombres G,Marta Tiene Un Marcapasos,1018992__2,[Verso 1]\nMarta tiene un marcapasos que le anima el cor...
6815,Grupo 2,1018992,Hombres G,Marta Tiene Un Marcapasos,1018992__2,Siento un golpe en el pecho\nYo solo quería besarte\nHa ...


## 2. The removal logs

A small helper keeps the per-file sections short. Each file is loaded, reduced to its distinct
`song_id`s, and intersected with the workbook. The direction of the expectation differs per file,
so the helper only reports — each section draws its own conclusion.

In [26]:
WORKBOOK_SONGS = set(assignment["song_id"])
SONG_LOOKUP = (
    assignment.drop_duplicates("song_id")
    .set_index("song_id")[["artist", "title", "group"]]
)


def load(name):
    # Lyrics fields hold embedded newlines; the default parser handles them, but the mixed-dtype
    # warning on the wider files is noise.
    return pd.read_csv(OLD / f"{name}.csv", low_memory=False)


def membership(frame, label):
    # Returns (songs in this file, the subset also present in the workbook) and prints the counts.
    songs = set(frame["song_id"].dropna().astype("int64"))
    present = songs & WORKBOOK_SONGS
    print(f"{label}: {len(frame):,} rows, {len(songs):,} distinct songs")
    print(f"  in the workbook    : {len(present):,}")
    print(f"  not in the workbook: {len(songs) - len(present):,}")
    return songs, present


def detail(song_ids, source=None, columns=("artist", "title")):
    # The workbook's own view of the given songs, so a leak can be eyeballed against the log.
    view = SONG_LOOKUP.loc[sorted(song_ids)].reset_index()
    if source is not None:
        view = view.merge(source, on="song_id", how="left", suffixes=("", "_log"))
    return view

### 2.1 `01_nan_lyrics.csv` — songs scraped with no lyrics

Genius returned an empty `lyrics` field for these. There is nothing to split into verses, so they
cannot legitimately reach the assignment.

In [27]:
nan_lyrics = load("01_nan_lyrics")
nan_songs, nan_present = membership(nan_lyrics, "01_nan_lyrics")

nan_lyrics[["artist_id", "artist", "song_id", "title", "language", "lyrics"]]

01_nan_lyrics: 6 rows, 6 distinct songs
  in the workbook    : 0
  not in the workbook: 6


,artist_id,artist,song_id,title,language,lyrics
0,405496,Bustamante,4242433,Héroes,NaN,NaN
1,201922,Sabina Ddumba,6896497,4 Life,NaN,NaN
2,201922,Patrik Jean,5267490,Losing Sleep (Acoustic Version),NaN,NaN
3,201922,Sabina Ddumba,6914871,No One Compares,NaN,NaN
4,201922,Sabina Ddumba,6914870,Sometimes,NaN,NaN
5,1307617,DAPS (ESP) & Kilvertz,10884503,Axelle,NaN,NaN


In [28]:
assert not nan_present, f"{len(nan_present)} songs with empty lyrics reached the assignment"
print("clean: none of the empty-lyrics songs is in the workbook")

clean: none of the empty-lyrics songs is in the workbook


**Clean.** All 6 are absent, as they should be.

### 2.2 `02_duplicates.csv` — the same song scraped under several artists

This file is the one place where a `song_id` in the log does *not* imply the song was dropped.
The rows here are duplicate *scrapes*: the same `song_id` collected once per `artist_id`. The
deduplication removed the redundant copies and kept one, so the song itself survives and the
`song_id` is expected in the workbook. What must **not** survive is more than one copy of it.

In [29]:
duplicates = load("02_duplicates")
dup_songs, dup_present = membership(duplicates, "02_duplicates")

print("\nrows per song_id in the log:")
print(duplicates["song_id"].value_counts().value_counts().rename("song_ids").to_frame())

02_duplicates: 64 rows, 31 distinct songs
  in the workbook    : 31
  not in the workbook: 0

rows per song_id in the log:
       song_ids
count          
2            30
4             1


So the check for this file is not absence but *multiplicity*: for each of these songs, the
workbook should hold exactly one verse per `verse_number`, in a single group.

In [30]:
dup_rows = assignment[assignment["song_id"].isin(dup_songs)]
repeated = dup_rows["verse_id"].duplicated().sum()
crossing = dup_rows.groupby("song_id")["group"].nunique().gt(1).sum()

print(f"workbook rows for these {len(dup_songs)} songs: {len(dup_rows):,}")
print(f"  repeated verse ids : {repeated}")
print(f"  songs split across groups: {crossing}")

assert repeated == 0 and crossing == 0, "a duplicate scrape survived into the assignment"
print("\nclean: deduplication held - one copy of each song, one group each")

workbook rows for these 31 songs: 276
  repeated verse ids : 0
  songs split across groups: 0

clean: deduplication held - one copy of each song, one group each


**Clean.** All 31 songs appear, each exactly once — which is the intended outcome, not a leak.

### 2.3 `03_manual_removed_languages.csv` — hand-removed non-Spanish songs

Songs inspected by hand and judged not to be Spanish (Korean, Turkish, Swedish, …). These were
removed from the corpus, so none of them should be in the assignment.

In [31]:
manual = load("03_manual_removed_languages")
manual_songs, manual_present = membership(manual, "03_manual_removed_languages")

print("\nlanguages in the log:")
print(manual["language"].value_counts().to_frame())

03_manual_removed_languages: 13 rows, 13 distinct songs
  in the workbook    : 13
  not in the workbook: 0

languages in the log:
          count
language       
sv           10
tr            2
ko            1


In [32]:
if manual_present:
    leaked = detail(manual_present,
                    manual[["song_id", "language"]].drop_duplicates("song_id"))
    print(f"LEAK: {len(leaked)} hand-removed songs are in the assignment\n")
    display(leaked)
    print("\nverses handed to annotators for these songs:",
          f"{assignment['song_id'].isin(manual_present).sum():,}")
else:
    print("clean: none of the hand-removed songs is in the workbook")

LEAK: 13 hand-removed songs are in the assignment



,song_id,artist,title,group,language
0,472208,Silvana Imam,I•M•A•M,Grupo 3,sv
1,883580,Ajda Pekkan,Bir Günah Gibi,Grupo 1,tr
2,2307147,Michel Dida,Höru Mej Bae (Remix),Grupo 2,sv
3,2908534,Mohammed Ali,Rapstjärna,Grupo 2,sv
4,3172547,Petter,Kall vinter,Grupo 2,sv
5,3284499,Sabina Ddumba,Vågorna,Grupo 3,sv
6,3400393,SHINee,혜야 (Y Si Fuera Ella),Grupo 1,ko
7,3429875,Sabina Ddumba,Orden I Vinden,Grupo 1,sv
8,3429876,Sabina Ddumba,Två Mörka Ögon,Grupo 1,sv
9,3527443,Sabina Ddumba,Varför Är Kärleken Röd,Grupo 1,sv



verses handed to annotators for these songs: 89


**Leak.** All 13 hand-removed songs are in the workbook, matching on `song_id`, `artist` *and*
`title` — these are the same records, not id collisions. Annotators are being shown Korean,
Turkish and Swedish lyrics.

### 2.4 `04_spanish_predicted_songs.csv` — the kept side of the language filter

The exception to the pattern. These are the songs the language classifier accepted as Spanish, so
the expectation inverts: they should be **present**. Anything here that is missing from the
workbook was dropped somewhere after the language filter.

In [33]:
spanish = load("04_spanish_predicted_songs")
spanish_songs, spanish_present = membership(spanish, "04_spanish_predicted_songs")

missing = spanish_songs - spanish_present
print(f"\naccepted-as-Spanish songs missing from the assignment: {len(missing)}")
print(f"workbook songs not covered by this file: {len(WORKBOOK_SONGS - spanish_songs):,}")

print("\nclassifier confidence on the accepted set:")
print(spanish["confidence"].describe().to_frame().T)

04_spanish_predicted_songs: 1,465 rows, 1,465 distinct songs
  in the workbook    : 1,465
  not in the workbook: 0

accepted-as-Spanish songs missing from the assignment: 0
workbook songs not covered by this file: 1,664

classifier confidence on the accepted set:
             count      mean      std       min       25%       50%       75%  \
confidence  1465.0  0.992118  0.01002  0.629455  0.992199  0.992995  0.993503   

                 max  
confidence  0.994186  


Every song in this file is in the workbook — the kept set passed through intact.

The reverse is not true, and is expected: the workbook holds 3,129 songs against this file's
1,465. This log covers only part of the corpus, so it certifies the songs it lists and says
nothing about the rest. It is a one-directional check.

### 2.5 `05_non_spanish_predicted_songs.csv` — the rejected side of the language filter

The mirror of the previous file: songs the classifier judged not to be Spanish, and so removed.
None should be in the assignment.

In [34]:
non_spanish = load("05_non_spanish_predicted_songs")
non_spanish_songs, non_spanish_present = membership(non_spanish, "05_non_spanish_predicted_songs")

print("\npredicted language of the rejected set:")
print(non_spanish["predict"].value_counts().head(10).to_frame())

05_non_spanish_predicted_songs: 101 rows, 101 distinct songs
  in the workbook    : 101
  not in the workbook: 0

predicted language of the rejected set:
         count
predict       
en          75
fr          14
it          12


In [35]:
if non_spanish_present:
    leaked = detail(non_spanish_present,
                    non_spanish[["song_id", "language", "predict", "confidence"]]
                    .drop_duplicates("song_id"))
    print(f"LEAK: {len(leaked)} classifier-rejected songs are in the assignment\n")
    display(leaked.head(15))
    print(f"\n... {len(leaked)} in total")
    print("verses handed to annotators for these songs:",
          f"{assignment['song_id'].isin(non_spanish_present).sum():,}")
    print("\nby predicted language:")
    display(non_spanish[non_spanish["song_id"].isin(non_spanish_present)]["predict"]
            .value_counts().to_frame())
else:
    print("clean: none of the rejected songs is in the workbook")

LEAK: 101 classifier-rejected songs are in the assignment



,song_id,artist,title,group,language,predict,confidence
0,230318,Katy Perry,Walking On Air,Grupo 2,en,en,0.842527
1,720556,F.A.M.,Forever Ain’t Long Enough,Grupo 3,en,en,0.971725
2,784446,Miguel Bosé,Bravi ragazzi,Grupo 3,it,it,0.995736
3,785119,Miguel Bosé,Se Tu Non Torni,Grupo 4,it,it,0.994390
4,793977,Miguel Bosé,Senza Di Te,Grupo 2,es,it,0.978271
5,865982,Mecano,Figlio della luna,Grupo 2,it,it,0.995281
6,949211,Julio Iglesias,My Love,Grupo 3,en,en,0.825427
7,961205,Julio Iglesias,Love Is On Our Side Again,Grupo 3,en,en,0.957093
8,995993,Julio Iglesias,Crazy,Grupo 2,en,en,0.922190
9,1175689,Julio Iglesias,Il faut toujours un perdant,Grupo 1,fr,fr,0.992567



... 101 in total
verses handed to annotators for these songs: 831

by predicted language:


,count
predict,
en,75
fr,14
it,12


**Leak.** All 101 classifier-rejected songs are in the workbook — the whole rejected set survived.

### 2.6 `06_all_eliminated_songs.csv` — the consolidated removal log

The union of the other removal files, carrying the full metadata for each dropped song. It is the
single list that the assignment should share no song with, so it is also the cleanest place to
size the problem.

First, confirm it really is the union — if it is not, one of the sections above tested a set this
file does not cover.

In [36]:
eliminated = load("06_all_eliminated_songs")
eliminated_songs, eliminated_present = membership(eliminated, "06_all_eliminated_songs")

union = nan_songs | dup_songs | manual_songs | non_spanish_songs
print(f"\nunion of files 01, 02, 03 and 05: {len(union):,} songs")
print(f"identical to this file: {union == eliminated_songs}")

06_all_eliminated_songs: 153 rows, 151 distinct songs
  in the workbook    : 145
  not in the workbook: 6

union of files 01, 02, 03 and 05: 151 songs
identical to this file: True


In [37]:
# Split the overlap by which log each song came from, since 02's presence is legitimate.
origin = pd.Series("", index=sorted(eliminated_songs), dtype="object")
for label, ids in [("01 empty lyrics", nan_songs),
                   ("02 duplicate scrape", dup_songs),
                   ("03 manual language", manual_songs),
                   ("05 predicted non-Spanish", non_spanish_songs)]:
    origin.loc[sorted(ids)] = label

summary = (
    pd.DataFrame({"origin": origin,
                  "in_assignment": origin.index.isin(WORKBOOK_SONGS)})
    .groupby("origin")["in_assignment"]
    .agg(songs="size", in_assignment="sum")
)
summary["expected_in_assignment"] = ["no", "yes (one copy)", "no", "no"]
summary

,songs,in_assignment,expected_in_assignment
origin,,,
01 empty lyrics,6,0,no
02 duplicate scrape,31,31,yes (one copy)
03 manual language,13,13,no
05 predicted non-Spanish,101,101,no


### What the overlap amounts to

In [38]:
genuine_leak = (manual_songs | non_spanish_songs) & WORKBOOK_SONGS
leaked_verses = assignment[assignment["song_id"].isin(genuine_leak)]

print(f"songs that should have been removed but are in the assignment: {len(genuine_leak)}")
print(f"verses of theirs handed to annotators: {len(leaked_verses):,} "
      f"({len(leaked_verses) / len(assignment):.2%} of the workbook)")
print("\nspread across the groups:")
print(leaked_verses.groupby("group").agg(songs=("song_id", "nunique"),
                                         verses=("verse_id", "size")))

songs that should have been removed but are in the assignment: 114
verses of theirs handed to annotators: 920 (3.83% of the workbook)

spread across the groups:
         songs  verses
group                 
Grupo 1     32     282
Grupo 2     24     197
Grupo 3     39     301
Grupo 4     19     140
